In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
import json, os, pathlib, subprocess, sys, uuid
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'
BRANCH='Content-V10'
EXPECTED_EXACT='60fedf82c14f14a196dfb350a7e743a59f870d09'
RUNNER_MODULE='experiments.run_content_v10_n96'
ATTEMPT_NONCE=uuid.uuid4().hex[:12]
SOURCE=pathlib.Path(f'/content/cegwm-content-v10-source-{ATTEMPT_NONCE}')
LOCAL=pathlib.Path(f'/content/Content-V10-60fedf8-local-{ATTEMPT_NONCE}')
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/CEG-WM/Content')
CAPTURE_LIMIT=4096

def _v10_git(source, *args):
    return subprocess.run(['git', *args], cwd=source, check=True, capture_output=True, text=True).stdout.strip()

def detach_v10_execution_checkout(source, branch, expected_exact):
    handoff_head = _v10_git(source, 'rev-parse', 'HEAD')
    if _v10_git(source, 'branch', '--show-current') != branch or _v10_git(source, 'status', '--porcelain'):
        raise RuntimeError('canonical handoff identity')
    if subprocess.run(['git','merge-base','--is-ancestor',expected_exact,handoff_head], cwd=source, capture_output=True).returncode != 0:
        raise RuntimeError('execution exact is not handoff ancestor')
    _v10_git(source, 'checkout', '--detach', expected_exact)
    if _v10_git(source, 'branch', '--show-current') != '' or _v10_git(source, 'rev-parse', 'HEAD') != expected_exact or _v10_git(source, 'status', '--porcelain'):
        raise RuntimeError('checkout identity')
    return handoff_head

def allocate_v10_attempt_paths(content_root, drive_root, attempt_nonce, now_utc):
    source = content_root / f'cegwm-content-v10-source-{attempt_nonce}'
    local = content_root / f'Content-V10-60fedf8-local-{attempt_nonce}'
    if source.exists() or local.exists():
        raise FileExistsError('create-only local path')
    while True:
        run_utc = now_utc().strftime('%Y%m%dT%H%M%S%fZ')
        drive_target = drive_root / f'Content-V10-60fedf8-{run_utc}'
        if not drive_target.exists():
            return source, local, drive_target, run_utc
PREFIX='CEGWM_CONTENT_V10_N96_PAIRED_SUMMARY '


In [ ]:
SOURCE, LOCAL, DRIVE_TARGET, RUN_UTC = allocate_v10_attempt_paths(pathlib.Path('/content'), DRIVE_ROOT, ATTEMPT_NONCE, lambda: datetime.now(timezone.utc))
subprocess.run(['git','clone','--no-single-branch','--branch',BRANCH,REPO_URL,str(SOURCE)],check=True)
HANDOFF_HEAD = detach_v10_execution_checkout(SOURCE, BRANCH, EXPECTED_EXACT)
def git(*args): return _v10_git(SOURCE, *args)
subprocess.run([sys.executable,'-m','pip','install',str(SOURCE)],check=True)
if git('rev-parse','HEAD')!=EXPECTED_EXACT or git('branch','--show-current') != '' or git('status','--porcelain') or LOCAL.exists() or DRIVE_TARGET.exists(): raise RuntimeError('post-install identity')
from google.colab import userdata
child_env={k:v for k,v in os.environ.items() if not any(x in k.upper() for x in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}; root_key=token=''
try:
    root_key=userdata.get('CEG_WM_ROOT_KEY'); token=userdata.get('HF_TOKEN'); child_env['CEG_WM_ROOT_KEY']=root_key; child_env['HF_TOKEN']=token
    p=subprocess.Popen([sys.executable,'-m',RUNNER_MODULE,'--repo-root',str(SOURCE),'--expected-exact',EXPECTED_EXACT,'--local-work-root',str(LOCAL),'--artifact-sink',str(DRIVE_TARGET)],cwd=SOURCE,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
finally:
    child_env.pop('CEG_WM_ROOT_KEY',None); child_env.pop('HF_TOKEN',None); root_key=token=''
summary=None; count=0
for raw in iter(p.stdout.readline,b''):
    if len(raw)>CAPTURE_LIMIT: raise RuntimeError('runner line bound')
    line=raw.decode('utf-8','strict').strip()
    if line.startswith(PREFIX): summary=json.loads(line[len(PREFIX):]); count+=1
rc=p.wait()
if count!=1 or not isinstance(summary,dict): raise RuntimeError('terminal summary')


In [ ]:
if rc==0 and summary.get('status')=='complete' and summary.get('completeness')=='complete' and summary.get('scientific_status')=='exploratory_evaluable' and summary.get('c1_committed_units')==32 and summary.get('c1_pair_count')==1056 and summary.get('n96_committed_units')==96 and summary.get('n96_failed_units')==0 and all(isinstance(summary.get(k),str) and summary[k] for k in ('result_path','sidecar_path','result_sha256')):
    print('CEGWM_CONTENT_V10_ARTIFACT '+json.dumps({'execution_exact':EXPECTED_EXACT,'result_path':summary['result_path'],'sidecar_path':summary['sidecar_path'],'result_sha256':summary['result_sha256'],'claim_ceiling':summary['claim_ceiling']},sort_keys=True,separators=(',',':')))
elif rc==2 and summary.get('status')=='incomplete' and summary.get('completeness')=='incomplete' and summary.get('scientific_status')=='not_evaluable' and summary.get('result_path') is None and summary.get('sidecar_path') is None and summary.get('result_sha256') is None:
    c1=summary.get('c1',{}); payload={'execution_exact':EXPECTED_EXACT,'status':'incomplete'}
    if c1.get('status')=='calibration_complete': payload['calibration_asset']=c1
    print('CEGWM_CONTENT_V10_INCOMPLETE '+json.dumps(payload,sort_keys=True,separators=(',',':')))
else: raise RuntimeError('runner completion contract')
